# ਪਾਠ 13 - ਏਜੰਟ ਯਾਦاشت


## ਸੈਟਅਪ

ਇਹ ਨੋਟਬੁੱਕ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ ਕਿਵੇਂ **Microsoft Agent Framework** (MAF) ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਪ੍ਰਤਿਸ਼ਠਿਤ ਮੈਮੋਰੀ** ਵਾਲਾ ਯਾਤਰਾ ਬੁੱਕਿੰਗ ਏਜੰਟ ਬਣਾਇਆ ਜਾਵੇ।

ਤੁਸੀਂ ਸਿੱਖੋਗੇ ਕਿ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀ ਏਜੰਟ ਮੈਮੋਰੀ — ਵਰਕਿੰਗ, ਛੋਟੀ ਮਿਆਦ, ਅਤੇ ਲੰਮੀ ਮਿਆਦ — ਕਿਸ ਤਰ੍ਹਾਂ ਏਜੰਟ ਨੂੰ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਜਾਣਕਾਰੀ ਰੱਖਣ ਅਤੇ ਇਸਤੇਮਾਲ ਕਰਨ 'ਤੇ ਪ੍ਰਭਾਵ ਪਾਉਂਦੀ ਹੈ।

**ਪੂਰਵ-ਆਪੇਖਿਆਵਾਂ:**
- ਇੱਕ Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਜਿਸ 'ਚ ਚੈਟ ਮਾਡਲ ਤਾਇਨਾਤ ਕੀਤਾ ਗਿਆ ਹੋਵੇ (ਜਿਵੇਂ `gpt-4o-mini`).
- Azure CLI ਨਾਲ ਲੌਗਇਨ ਕੀਤਾ ਹੋਵੇ — ਆਪਣੀ ਟਰਮੀਨਲ ਵਿੱਚ `az login` ਚਲਾਓ।
- `AZURE_AI_PROJECT_ENDPOINT` — ਤੁਹਾਡੇ Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਦਾ ਐਂਡਪਇੰਟ।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ਤੁਹਾਡੇ ਤਾਇਨਾਤ ਕੀਤੇ ਮਾਡਲ ਦਾ ਨਾਮ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ਏਜੰਟ ਮੈਮੋਰੀ ਦੇ ਕਿਸਮ

ਏਆਈ ਏਜੰਟ ਵੱਖ-ਵੱਖ ਕਿਸਮ ਦੀਆਂ ਮੈਮੋਰੀਆਂ ਦੀ ਵਰਤੋਂ ਕਰ ਸਕਦੇ ਹਨ, ਹਰ ਇੱਕ ਦਾ ਇੱਕ ਵੱਖਰਾ ਉਦੇਸ਼ ਹੁੰਦਾ ਹੈ:

### ਵਰਕਿੰਗ ਮੈਮੋਰੀ
ਗੱਲਬਾਤ ਦੀ ਸੂਤੀ — ਇੱਕ ਸੈਸ਼ਨ ਵਿੱਚ ਬਦਲੇ ਗਏ ਸੁਨੇਹੇ। ਏਜੰਟ ਇਸੇ ਥ੍ਰੈਡ ਦੇ ਪਹਿਲਾਂ ਦੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਸਹੀਤਰਤੀਬੀ ਬਨਾਈ ਰੱਖਣ ਲਈ ਮੁੜ ਦੇਖ ਸਕਦਾ ਹੈ। MAF ਵਿੱਚ ਇਹ **`agent.create_session()`** ਰਾਹੀਂ ਬਣਾਈ ਜਾਂਦੀ ਹੈ, ਜੋ `AgentSession` ਵਾਪਸ ਕਰਦੀ ਹੈ।

### ਸ਼ਾਰਟ-ਟਰਮ ਮੈਮੋਰੀ
ਉਹ ਜਾਣਕਾਰੀ ਜੋ ਕਿ ਕੰਮ ਜਾਂ ਸੈਸ਼ਨ ਦੀ ਮਿਆਦ ਵਾਸਤੇ ਟਿਕਦੀ ਹੈ ਪਰ ਹਮੇਸ਼ਾ ਲਈ ਸਟੋਰ ਨਹੀਂ ਹੁੰਦੀ। ਉਦਾਹਰਨ ਵਜੋਂ, ਏਜੰਟ ਕਈ ਦੀਵਾਰ ਤੱਕ ਚੱਲ ਰਹੀ ਯੋਜਨਾ ਬਣਾਉਣ ਵਾਲੀ ਗੱਲਬਾਤ ਦੌਰਾਨ ਤੱਥ ਇਕੱਠੇ ਕਰ ਸਕਦਾ ਹੈ ਅਤੇ ਉਹਨਾਂ ਨੂੰ ਅੰਤਿਮ ਯਾਤਰਾ ਤਯਾਰ ਕਰਨ ਲਈ ਵਰਤ ਸਕਦਾ ਹੈ।

### ਲੰਬੇ ਸਮੇਂ ਦੀ ਮੈਮੋਰੀ
ਵਰਜਨ ਅਤੇ ਤੱਥ ਜੋ ਕਿ **ਸੈਸ਼ਨਾਂ ਦੇ ਪਾਰ** ਟਿਕੇ ਰਹਿੰਦੇ ਹਨ। ਵਾਪਸ ਆਉਣ ਵਾਲੇ ਯੂਜ਼ਰ ਨੂੰ ਆਪਣੀਆਂ ਡਾਈਟਰੀ ਪਾਬੰਦੀਆਂ ਜਾਂ ਯਾਤਰਾ ਦੇ ਅੰਦਾਜ ਨੂੰ ਦੁਹਰਾਉਣ ਦੀ ਲੋੜ ਨਹੀਂ ਹੋਣੀ ਚਾਹੀਦੀ। ਲੰਬੇ ਸਮੇਂ ਦੀ ਮੈਮੋਰੀ ਆਮ ਤੌਰ 'ਤੇ ਬਾਹਰੀ ਸਟੋਰ ਨਾਲ ਸਮਰਥਿਤ ਹੁੰਦੀ ਹੈ — ਇਕ ਡਾਟਾਬੇਸ, ਫਾਈਲ ਜਾਂ ਵੈਕਟਰ ਇੰਡੈਕਸ — ਅਤੇ ਇਹ ਏਜੰਟ ਕੋਲ ਟੂਲਜ਼ ਰਾਹੀਂ ਪੇਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ।


## ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਵਿਥ ਸੈਸ਼ਨਜ਼

ਸਮਝਦਾਰੀ ਦੀ ਸਭ ਤੋਂ ਸਾਦਾ ਰੂਪ ਗੱਲਬਾਤ ਸੈਸ਼ਨ ਹੁੰਦੀ ਹੈ। ਜਦੋਂ ਤੁਸੀਂ ਇੱਕੋ ਸੈਸ਼ਨ (`agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਨੂੰ ਲਗਾਤਾਰ `agent.run()` ਕਾਲਾਂ ਵਿੱਚ ਪਾਸ ਕਰਦੇ ਹੋ, ਤਾਂ ਏਜੰਟ ਉਸ ਗੱਲਬਾਤ ਦਾ ਪੂਰਾ ਇਤਿਹਾਸ ਦੇਖਦਾ ਹੈ ਅਤੇ ਪਹਿਲਾਂ ਦੀਆਂ ਜਾਣਕਾਰੀਆਂ ਯਾਦ ਕਰ ਸਕਦਾ ਹੈ।

ਆਓ ਇੱਕ ਯਾਤਰਾ ਏਜੰਟ ਬਣਾਈਏ ਅਤੇ ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਨੂੰ ਦਰਸਾਈਏ।


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ਏਜੰਟ ਨੇ ਬਜਟ ਨੂੰ ਸਹੀ ਤਰ੍ਹਾਂ ਯਾਦ ਕੀਤਾ ਕਿਉਂਕਿ ਦੋਹਾਂ ਸੁਨੇਹੇ ਇਕੋ ਹੀ ਸੈਸ਼ਨ ਸਾਂਝੇ ਕਰਦੇ ਹਨ। ਇਹ **ਕਾਮ ਕਰਦੀ ਮੈਮੋਰੀ** ਹੈ — ਇਹ ਸਿਰਫ ਸੈਸ਼ਨ ਦੀ ਮਿਆਦ ਲਈ ਮੌਜੂਦ ਹੁੰਦੀ ਹੈ।

### ਨਵੀਂ ਥ੍ਰੈਡ ਨਾਲ ਕੀ ਹੁੰਦਾ ਹੈ?

ਜੇ ਅਸੀਂ ਇੱਕ **ਨਵਾਂ** ਸੈਸ਼ਨ ਬਣਾਈਏ, ਤਾਂ ਏਜੰਟ ਨੂੰ ਪਿਛਲੀ ਗੱਲਬਾਤ ਦੀ ਕੋਈ ਯਾਦ ਨਹੀਂ ਹੁੰਦੀ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ਲੰਬੇ ਸਮੇਂ ਦੀ ਯਾਦਸ਼ਕਤੀ ਦਾ ਨਮੂਨਾ

ਉਪਭੋਗਤਾ ਦੀਆਂ ਪਸੰਦਾਂ ਨੂੰ **ਸੈਸ਼ਨਾਂ ਦੇ ਅੰਦਰ** ਯਾਦ ਰੱਖਣ ਲਈ, ਸਾਨੂੰ ਇੱਕ ਸਥਾਈ ਸਟੋਰ ਦੀ ਲੋੜ ਹੈ ਜੋ ਗੱਲਬਾਤ ਦੇ ਧਾਗੇ ਦੇ ਬਾਹਰ ਵੱਸਦਾ ਹੈ। ਏਜੰਟ ਇਸ ਸਟੋਰ ਤਕ ਪਹੁੰਚਦਾ ਹੈ **ਟੂਲਜ਼** ਰਾਹੀਂ — ਫੰਕਸ਼ਨ ਜੋ ਇਹ ਸੂਚਨਾ ਨੂੰ ਸੇਵ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਸਧਾਰਨ ਇਨ-ਮੇਮੋਰੀ ਪਸੰਦ ਸਟੋਰ ਲਾਗੂ ਕਰਦੇ ਹਾਂ (ਉਤਪਾਦਨ ਵਿੱਚ ਤੁਸੀਂ ਇਸਨੂੰ ਡੈਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਸੂਚਕਾਂਕ ਨਾਲ ਬੈਕ ਕਰੋਗੇ) ਅਤੇ ਇਸਨੂੰ ਟੂਲਜ਼ ਵਜੋਂ ਐਜੰਟ ਲਈ ਉਪਲਬਧ ਕਰਵਾਉਂਦੇ ਹਾਂ।

### ਆਰਕੀਟੈਕਚਰ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — ਪਹਿਲੀ ਵਾਰੀ ਉਪਭੋਗਤਾ ਵਰ੍ਹੇਗੱਡੀ ਦੀ ਯਾਤਰਾ ਬੁੱਕ ਕਰਦਾ ਹੈ

ਸਾਰਾਹ ਪਹਿਲੀ ਵਾਰੀ ਆਉਂਦੀ ਹੈ। ਏਜੰਟ ਨੂੰ ਉਸ ਦੀਆਂ ਪਸੰਦਾਂ ਟੂਲਸ ਰਾਹੀਂ ਸਟੋਰ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਉਹਨਾਂ ਨੂੰ ਹੋਟਲਾਂ ਦੀ ਸਿਫਾਰਸ਼ ਲਈ ਵਰਤਣਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ਸਿਨੇਰੀਓ 2 — ਸਾਰਾਹ ਹਫ਼ਤੇ ਬਾਅਦ ਵਾਪਸ ਆਉਂਦੀ ਹੈ

ਸਾਰਾਹ ਇੱਕ **ਨਵੀਂ ਥ੍ਰੈੱਡ** ਸ਼ੁਰੂ ਕਰਦੀ ਹੈ (ਨਵਾਂ ਸੈਸ਼ਨ ਨਕਲ ਕਰਦੇ ਹੋਏ)। ਵਰਕਿੰਗ ਮੈਮੋਰੀ ਖਾਲੀ ਹੈ, ਪਰ ਲੰਬੇ ਸਮੇਂ ਦੀ ਪ੍ਰਾਥਮਿਕਤਾ ਸਟੋਰ ਵਿੱਚ ਅਜੇ ਵੀ ਉਸਦੀ ਜਾਣਕਾਰੀ ਹੈ। ਏਜੰਟ ਨੂੰ ਇਹ ਪ੍ਰਾਪਤ ਕਰਨੀ ਚਾਹੀਦੀ ਹੈ ਅਤੇ ਇਸਦੀ ਵਰਤੋਂ ਕਰਕੇ ਸਿਫ਼ਾਰਸ਼ਾਂ ਨੂੰ ਨਿੱਜੀਕਰਣ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ਸਰੰਸ਼

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਤਿੰਨ ਕਿਸਮਾਂ ਦੀ ਏਜੰਟ ਮੈਮੋਰੀ ਬਾਰੇ ਸਿੱਖਿਆ ਅਤੇ ਇਹਨਾਂ ਨੂੰ Microsoft Agent Framework ਨਾਲ ਕਿਵੇਂ ਲਾਗੂ ਕਰਨਾ ਹੈ:

| ਮੈਮੋਰੀ ਕਿਸਮ | MAF ਮਕੈਨਿਜ਼ਮ | ਆਜੀਵਿਕਾਲ |
|---|---|---|
| **ਕਾਮ ਕਰਨ ਵਾਲੀ** | `agent.create_session()` | ਇੱਕ ਸਿੰਗਲ ਗੱਲਬਾਤ |
| **ਛੋਟੀ ਅਵਧੀ ਵਾਲੀ** | ਇੱਕ ਧਾਗੇ ਵਿੱਚ ਇਕੱਠੀ ਕੀਤੀ ਸੰਦਰਭ | ਇੱਕ ਸਿੰਗਲ ਕਾਰਜ / ਸੈਸ਼ਨ |
| **ਲੰਬੀ ਅਵਧੀ ਵਾਲੀ** | ਬਾਹਰੀ ਸਟੋਰ ਜਿਨ੍ਹਾਂ ਨੂੰ `@tool` ਫੰਕਸ਼ਨਾਂ ਦੁਆਰਾ ਪ੍ਰਾਪਤ ਕੀਤਾ ਜਾਂਦਾ ਹੈ | ਸੈਸ਼ਨਾਂ ਦੇ ਦੌਰਾਨ |

### ਮੁੱਖ ਗੱਲਾਂ
1. **`agent.create_session()`** ਕੰਮ ਕਰਨ ਵਾਲੀ ਮੈਮੋਰੀ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ — ਏਜੰਟ ਸੈਸ਼ਨ ਵਿੱਚ ਪੂਰੀ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਵੇਖਦਾ ਹੈ।
2. **ਨਵੇਂ ਸੈਸ਼ਨ ਸੰਦਰਭ ਗੁਆ ਦੇਂਦੇ ਹਨ** — ਲੰਬੀ ਅਵਧੀ ਵਾਲੀ ਮੈਮੋਰੀ ਤੋਂ ਬਿਨਾਂ ਏਜੰਟ ਪਿਛਲੀਆਂ ਗੱਲਬਾਤਾਂ ਨੂੰ ਯਾਦ ਨਹੀਂ ਕਰ ਸਕਦਾ।
3. **`@tool` ਫੰਕਸ਼ਨ** ਫੰਕਸ਼ਨਾਂ ਇਹ ਖਾਈ ਪੂਰੀ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਨੂੰ ਜਾਣਕਾਰੀ ਸਟੋਰ ਕਰਨ ਅਤੇ ਪ੍ਰਾਪਤ ਕਰਨ ਦੀ ਆਗਿਆ ਦਿੰਦੇ ਹਨ।
4. **ਵੈਯਕਤੀਕਰਨ ਸਮੇਂ ਨਾਲ ਸੁਧਰਦਾ ਹੈ** — ਜਿੰਨਾ ਵੱਧ ਪਸੰਦਾਂ ਸੰਗ੍ਰਹਿਤ ਕੀਤੀਆਂ ਜਾਂਦੀਆਂ ਹਨ, ਓਨਾ ਚੰਗੀ ਏਜੰਟ ਦੀ ਸਿਫਾਰਸ਼ਾਂ ਹੁੰਦੀਆਂ ਹਨ।

### ਅਸਲੀ ਦੁਨੀਆ ਵਿੱਚ ਪ੍ਰਯੋਗ
- **ਗਾਹਕ ਸੇਵਾ**: ਗਾਹਕ ਦਾ ਇਤਿਹਾਸ ਅਤੇ ਪਸੰਦਾਂ ਯਾਦ ਰੱਖੋ
- **ਨਿੱਜੀ ਸਹਾਇਕ**: ਦਿਨਾਂ ਜਾਂ ਹਫਤਿਆਂ ਦੇ ਦੌਰਾਨ ਸੰਦਰਭ ਬਣਾਈ ਰੱਖੋ
- **ਸਿਹਤ ਸੇਵਾ**: ਮਰੀਜ਼ ਦੀ ਜਾਣਕਾਰੀ ਅਤੇ ਪਸੰਦਾਂ ਦਾ ਟ੍ਰੈਕ ਰੱਖੋ
- **ਈ-ਕਾਮਰਸ**: ਇਤਿਹਾਸ ਦੇ ਆਧਾਰ 'ਤੇ ਵੈਯਕਤੀਕ੍ਰਿਤ ਖਰੀਦਦਾਰੀ

### ਅਗਲੇ ਕਦਮ
- ਮੈਮੋਰੀ ਡਿਕਸ਼ਨਰੀ ਦੀ ਝਾੜ ਨੂੰ ਡੇਟਾਬੇਸ ਜਾਂ ਵੇਕਟਰ ਸਟੋਰ (ਜਿਵੇਂ Azure AI Search) ਨਾਲ ਬਦਲੋ
- ਸਮੇਂ-ਸੰਵੇਦਨਸ਼ੀਲ ਜਾਣਕਾਰੀ ਲਈ ਮੈਮੋਰੀ ਮਿਆਦ ਸ਼ੁਰੂ ਕਰੋ
- ਸਾਂਝੀ ਮੈਮੋਰੀ ਨਾਲ ਬਹੁ-ਏਜੰਟ ਸਿਸਟਮ ਬਣਾਓ
- ਗਿਆਨ-ਗ੍ਰਾਫ-ਸਹਾਇਤ ਮੈਮੋਰੀ ਲਈ Cognee ਨੋਟਬੁੱਕ ਦੀ ਜਾਂਚ ਕਰੋ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਝੂਠ ਬਿਆਨ**:  
ਇਸ ਦਸਤਾਵੇਜ਼ ਨੂੰ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸ਼ੁੱਧਤਾ ਲਈ ਯਤਨਸ਼ীল ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਇਹ ਗੱਲ ਧਿਆਨ ਵਿੱਚ ਰੱਖੋ ਕਿ ਸੁਤੰਤਰ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਪ੍ਰામਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਣ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ਾਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਭੁੱਲਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
